# Policy Area Diagnostic

Diagnose policy area coverage in the revamp pipeline vs. the original thesis.

**Key question:** The revamp uses only committee-based mapping (no bill-classifier fallback).
Is coverage sufficient for the GLMM replication?

## Background

The original thesis assigned policy areas using a two-tier approach:
1. **Primary:** Map each granule's committee metadata to one of 21 CAP policy domains (~45 committees)
2. **Fallback:** For granules without committee data, use a Stanford/USA Facts bill classifier
3. After both methods, ~1/3 of mentions still had no policy area and were dropped

The revamp uses only the committee-based mapping (no bill-classifier fallback) but has a
larger committee dictionary (85+ entries vs the thesis's ~45).

**This notebook does NOT modify any existing pipeline scripts.**

## Part 1: Diagnose Current Coverage

### 1a: level1.csv

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

level1 = pd.read_csv('../../data/output/level1.csv', low_memory=False)

# Check for policy area columns
policy_cols = [c for c in level1.columns if 'issue_area' in c.lower() or 'policy' in c.lower()]
print('POLICY AREA COVERAGE IN level1.csv')
print('=' * 50)

if 'issue_area' in level1.columns:
    total = len(level1)
    with_policy = level1['issue_area'].notna().sum()
    without_policy = total - with_policy
    n_unique = level1['issue_area'].nunique()
    print(f'Column name:         issue_area / issue_area_name')
    print(f'Total rows:          {total:,}')
    print(f'Rows with policy:    {with_policy:,} ({with_policy/total*100:.1f}%)')
    print(f'Rows without:        {without_policy:,} ({without_policy/total*100:.1f}%)')
    print(f'Unique policy areas: {n_unique}')
    print()
    print('Value counts (issue_area_name):')
    print(level1['issue_area_name'].value_counts().to_string())
else:
    print('NO issue_area column found!')
    print(f'Available policy-related columns: {policy_cols}')

POLICY AREA COVERAGE IN level1.csv
Column name:         issue_area / issue_area_name
Total rows:          53,892
Rows with policy:    14,390 (26.7%)
Rows without:        39,502 (73.3%)
Unique policy areas: 18

Value counts (issue_area_name):
issue_area_name
Defense                  3333
Macroeconomics           3276
Energy                   1771
Government Operations    1038
Agriculture               966
Domestic Commerce         962
Law and Crime             800
Health                    591
Environment               336
Transportation            316
Technology                313
Labor                     247
Civil Rights              176
International Affairs     122
Education                  51
Housing                    48
Public Lands               31
Social Welfare             13


### 1b: analysis_dataset_replication.csv

In [2]:
df = pd.read_csv('../../data/output/analysis_dataset_replication.csv', low_memory=False)

print('POLICY AREA COVERAGE IN analysis_dataset_replication.csv')
print('=' * 60)

total = len(df)
with_policy = df['issue_area'].notna().sum()
without_policy = total - with_policy

# Separate zero-mention padding rows
is_zero = (df['is_zero_mention'] == 1) if 'is_zero_mention' in df.columns else pd.Series(False, index=df.index)
mentions = df[~is_zero]
m_with = mentions['issue_area'].notna().sum()
m_total = len(mentions)

print(f'Total rows:               {total:,}')
print(f'  Zero-mention padding:   {is_zero.sum():,}')
print(f'  Actual mentions:        {m_total:,}')
print()
print(f'All rows with policy:     {with_policy:,} / {total:,} ({with_policy/total*100:.1f}%)')
print(f'Mentions with policy:     {m_with:,} / {m_total:,} ({m_with/m_total*100:.1f}%)')
print(f'Mentions without policy:  {m_total - m_with:,} ({(m_total - m_with)/m_total*100:.1f}%)')
print(f'Unique policy areas:      {df["issue_area"].nunique()}')
print()
print('Value distribution:')
print(df['issue_area_name'].value_counts().to_string())

POLICY AREA COVERAGE IN analysis_dataset_replication.csv
Total rows:               57,073
  Zero-mention padding:   3,181
  Actual mentions:        53,892

All rows with policy:     14,390 / 57,073 (25.2%)
Mentions with policy:     14,390 / 53,892 (26.7%)
Mentions without policy:  39,502 (73.3%)
Unique policy areas:      18

Value distribution:
issue_area_name
Defense                  3333
Macroeconomics           3276
Energy                   1771
Government Operations    1038
Agriculture               966
Domestic Commerce         962
Law and Crime             800
Health                    591
Environment               336
Transportation            316
Technology                313
Labor                     247
Civil Rights              176
International Affairs     122
Education                  51
Housing                    48
Public Lands               31
Social Welfare             13


### 1c: Committee-Policy Mapping

The mapping is currently duplicated in 3-4 files. We check the primary one.

In [3]:
import sys, json

# Load from canonical JSON reference
json_path = '../../data/input/committee_policy_mapping.json'
try:
    with open(json_path, encoding='utf-8') as f:
        ref = json.load(f)
    mapping_ref = ref['mapping']
    print(f'Mapping loaded from {json_path}')
    print(f'Total committee-to-policy mappings: {len(mapping_ref)}')
    print()
    print(f'{"Committee":60s} {"Policy Area":25s} {"CAP Code"}')
    print('-' * 95)
    for comm, info in sorted(mapping_ref.items()):
        area = info.get("policy_area", "N/A") or "(skip)"
        code = info.get("cap_code", "N/A") or "-"
        print(f'{comm:60s} {str(area):25s} {code}')
except FileNotFoundError:
    print(f'{json_path} not found.')
    print('Falling back to import...')
    sys.path.insert(0, '../../interest_group_analysis/2.data_processing')
    from committee_policy_linkage import COMMITTEE_TO_POLICY
    print(f'Loaded {len(COMMITTEE_TO_POLICY)} entries from committee_policy_linkage.py')

Mapping loaded from ../data/input/committee_policy_mapping.json
Total committee-to-policy mappings: 85

Committee                                                    Policy Area               CAP Code
-----------------------------------------------------------------------------------------------
Ad Hoc Committee on Energy                                   Energy                    800
Committee of the Whole House on the State of the Union       (skip)                    -
Committee on Agriculture                                     Agriculture               400
Committee on Agriculture and Forestry                        Agriculture               400
Committee on Agriculture, Nutrition, and Forestry            Agriculture               400
Committee on Appropriations                                  Macroeconomics            100
Committee on Armed Services                                  Defense                   1600
Committee on Banking and Currency                            Domesti

### 1d: Granule Committee Data

The raw granule metadata has committee information spread across per-package files.
We check how many level1 granules actually have committee data.

In [4]:
# Load all granule_committees.csv files
files = glob.glob('../../data/intermediate/normalized_114/by_package/*/granule_committees.csv')
print(f'Committee data files found: {len(files)}')

level1_granules = set(level1['granuleId'].unique())
print(f'Unique granules in level1: {len(level1_granules):,}')

# Scan all files
granules_in_files = set()
granules_with_name = set()
all_committee_names = set()

for f in files:
    cdf = pd.read_csv(f)
    overlap = set(cdf['granuleId'].unique()) & level1_granules
    granules_in_files.update(overlap)
    mask = cdf['committeeName'].notna() & cdf['granuleId'].isin(level1_granules)
    granules_with_name.update(cdf.loc[mask, 'granuleId'].unique())
    all_committee_names.update(cdf.loc[cdf['committeeName'].notna(), 'committeeName'].unique())

not_in_files = level1_granules - granules_in_files
in_files_no_name = granules_in_files - granules_with_name

print()
print('GRANULE COMMITTEE DATA AVAILABILITY (for level1 granules)')
print('=' * 60)
print(f'Granules in committee files:       {len(granules_in_files):,} ({len(granules_in_files)/len(level1_granules)*100:.1f}%)')
print(f'  With a committee name:           {len(granules_with_name):,} ({len(granules_with_name)/len(level1_granules)*100:.1f}%)')
print(f'  In files but no committee name:  {len(in_files_no_name):,} ({len(in_files_no_name)/len(level1_granules)*100:.1f}%)')
print(f'Not in any committee file:         {len(not_in_files):,} ({len(not_in_files)/len(level1_granules)*100:.1f}%)')
print()
print(f'Unique committee names in data: {len(all_committee_names)}')

# How many mentions come from granules with committee data?
level1['has_committee'] = level1['granuleId'].isin(granules_with_name)
m_with_comm = level1['has_committee'].sum()
m_total = len(level1)
print(f'Mentions from granules with committee: {m_with_comm:,} / {m_total:,} ({m_with_comm/m_total*100:.1f}%)')
print(f'Mentions from granules without:        {m_total - m_with_comm:,} ({(m_total - m_with_comm)/m_total*100:.1f}%)')

Committee data files found: 371
Unique granules in level1: 10,657



GRANULE COMMITTEE DATA AVAILABILITY (for level1 granules)
Granules in committee files:       4,694 (44.0%)
  With a committee name:           1,500 (14.1%)
  In files but no committee name:  3,194 (30.0%)
Not in any committee file:         5,963 (56.0%)

Unique committee names in data: 77
Mentions from granules with committee: 14,404 / 53,892 (26.7%)
Mentions from granules without:        39,488 (73.3%)


#### Unmatched Committees

Check which committee names in the data are NOT covered by the mapping dictionary.

In [5]:
# Check which committee names don't match the mapping
# Load mapping keys from JSON reference
try:
    with open('../../data/input/committee_policy_mapping.json', encoding='utf-8') as f:
        ref_data = json.load(f)
    mapping_keys = list(ref_data['mapping'].keys())
    mapping_source = 'committee_policy_mapping.json'
except FileNotFoundError:
    mapping_keys = []
    mapping_source = 'UNAVAILABLE'

print(f'Mapping source: {mapping_source}')
print(f'Mapping size: {len(mapping_keys)} entries')
print()

# Test each committee name against mapping (uses substring match, same as pipeline)
matched_names = set()
unmatched_names = set()
for name in all_committee_names:
    found = False
    for key in mapping_keys:
        if key.lower() in name.lower():
            found = True
            break
    if found:
        matched_names.add(name)
    else:
        unmatched_names.add(name)

print(f'Committee names matched by mapping: {len(matched_names)} / {len(all_committee_names)}')
print(f'Committee names NOT matched:        {len(unmatched_names)}')
print()
if unmatched_names:
    print('UNMATCHED COMMITTEE NAMES:')
    for n in sorted(unmatched_names):
        print(f'  - {n}')
else:
    print('All committee names in the data are covered by the mapping.')

Mapping source: committee_policy_mapping.json
Mapping size: 85 entries

Committee names matched by mapping: 77 / 77
Committee names NOT matched:        0

All committee names in the data are covered by the mapping.


### 1e: Legacy Coverage

In [6]:
legacy = pd.read_csv('../../data/output/data_legacy_thesis.txt', low_memory=False)
print(f'Legacy dataset: {len(legacy):,} rows, {legacy.shape[1]} columns')
print()

# Key policy columns in legacy
policy_cols = {
    'level1_committee_policy_area': 'Committee-based policy (primary)',
    'level1_committee_policy_num': 'Committee-based CAP code',
    'level1_bill_policy_num': 'Bill-classifier policy (fallback)',
    'level1_policy_area_x': 'Bill-classifier policy name',
    'level1_issue_area': 'Final combined issue area (CAP code)',
}

print('LEGACY POLICY AREA COVERAGE')
print('=' * 60)
for col, desc in policy_cols.items():
    if col in legacy.columns:
        nn = legacy[col].notna().sum()
        print(f'{desc:45s}: {nn:,} / {len(legacy):,} ({nn/len(legacy)*100:.1f}%)')

print()
print('INTERPRETATION:')
# Committee-only coverage in legacy
comm_cov = legacy['level1_committee_policy_area'].notna().sum()
bill_cov = legacy['level1_bill_policy_num'].notna().sum()
combined = legacy['level1_issue_area'].notna().sum()
total = len(legacy)
print(f'Committee-only coverage: {comm_cov:,} ({comm_cov/total*100:.1f}%)')
print(f'Bill-classifier coverage: {bill_cov:,} ({bill_cov/total*100:.1f}%)')
print(f'Combined (union):         {combined:,} ({combined/total*100:.1f}%)')
print(f'Missing (dropped):        {total - combined:,} ({(total - combined)/total*100:.1f}%)')
print()
print(f'The bill classifier added ~{combined - comm_cov:,} observations ({(combined - comm_cov)/total*100:.1f}%)')
print(f'that the committee mapping alone could not cover.')

Legacy dataset: 22,414 rows, 175 columns

LEGACY POLICY AREA COVERAGE
Committee-based policy (primary)             : 12,049 / 22,414 (53.8%)
Committee-based CAP code                     : 12,049 / 22,414 (53.8%)
Bill-classifier policy (fallback)            : 13,005 / 22,414 (58.0%)
Bill-classifier policy name                  : 13,005 / 22,414 (58.0%)
Final combined issue area (CAP code)         : 14,392 / 22,414 (64.2%)

INTERPRETATION:
Committee-only coverage: 12,049 (53.8%)
Bill-classifier coverage: 13,005 (58.0%)
Combined (union):         14,392 (64.2%)
Missing (dropped):        8,022 (35.8%)

The bill classifier added ~2,343 observations (10.5%)
that the committee mapping alone could not cover.


In [7]:
# Legacy issue_area distribution
print('Legacy issue_area distribution (CAP codes):')
print(legacy['level1_issue_area'].value_counts().to_string())

Legacy issue_area distribution (CAP codes):
level1_issue_area
100.0     2877
2000.0    2072
1600.0    1550
1200.0    1434
1500.0    1369
800.0     1135
600.0     1012
400.0      832
300.0      579
700.0      299
1900.0     263
2100.0     243
1700.0     223
200.0      131
1400.0     105
1300.0     100
1000.0      82
500.0       39
900.0       26
2300.0      18
1800.0       3


## Part 2: Gap Assessment

In [8]:
print('POLICY AREA GAP ASSESSMENT')
print('=' * 60)

# Revamp stats
revamp_total = len(level1)
revamp_with = level1['issue_area'].notna().sum()
revamp_pct = revamp_with / revamp_total * 100

# Legacy stats
legacy_total = len(legacy)
legacy_combined = legacy['level1_issue_area'].notna().sum()
legacy_pct_before = legacy_combined / legacy_total * 100

print(f'Revamp coverage:    {revamp_pct:.1f}% of mentions have a policy area')
print(f'Legacy coverage:    {legacy_pct_before:.1f}% of mentions had a policy area (before dropping)')
print(f'Legacy after drops: 100.0% (missing rows were dropped)')
print()

missing_pct = 100 - revamp_pct
if missing_pct < 10:
    severity = 'MINIMAL (<10% missing)'
    recommendation = 'Document and proceed. Drop missing rows as thesis did.'
elif missing_pct < 30:
    severity = 'MODERATE (10-30% missing)'
    recommendation = 'A fallback is needed. Consider bill-based policy area or drop.'
else:
    severity = 'SEVERE (>30% missing)'
    recommendation = 'Committee-only approach insufficient. Fallback or methodology change needed.'

print(f'Gap severity: {severity}')
print(f'Recommendation: {recommendation}')
print()

print('ROOT CAUSE ANALYSIS')
print('-' * 60)
print(f'Total granules in level1:        {len(level1_granules):,}')
print(f'Granules with committee names:   {len(granules_with_name):,} ({len(granules_with_name)/len(level1_granules)*100:.1f}%)')
print(f'Granules in files but no name:   {len(in_files_no_name):,}')
print(f'Granules not in committee files:  {len(not_in_files):,}')
print()
print('Most Congressional Record granules (speeches) are NOT associated with')
print('a committee in the GovInfo API metadata. The committee field is only')
print('populated for speeches given in committee hearings or referencing')
print('committee business. Floor speeches, tributes, and general debate')
print('typically have no committee tag.')
print()
print('This is why the thesis used a BILL CLASSIFIER as a fallback - it')
print('inferred policy area from the bills discussed in the speech, even')
print('when no committee metadata was present.')

POLICY AREA GAP ASSESSMENT
Revamp coverage:    26.7% of mentions have a policy area
Legacy coverage:    64.2% of mentions had a policy area (before dropping)
Legacy after drops: 100.0% (missing rows were dropped)

Gap severity: SEVERE (>30% missing)
Recommendation: Committee-only approach insufficient. Fallback or methodology change needed.

ROOT CAUSE ANALYSIS
------------------------------------------------------------
Total granules in level1:        10,657
Granules with committee names:   1,500 (14.1%)
Granules in files but no name:   3,194
Granules not in committee files:  5,963

Most Congressional Record granules (speeches) are NOT associated with
a committee in the GovInfo API metadata. The committee field is only
populated for speeches given in committee hearings or referencing
committee business. Floor speeches, tributes, and general debate
typically have no committee tag.

This is why the thesis used a BILL CLASSIFIER as a fallback - it
inferred policy area from the bills dis

## Part 3: Evaluate Fallback Options

Since coverage is SEVERE, we need to evaluate options for filling the gap.

### Option A: Congress.gov Bill Policy Areas

The pipeline already links mentions to bills via `2.bills_linkage.py`.
The Congress.gov API returns a `policyArea` field on bills. Can we use this?

In [9]:
# Check bill reference data availability
ref_files = glob.glob('../../data/intermediate/normalized_114/by_package/*/granule_references.csv')
print(f'Granule reference files: {len(ref_files)}')

# Count granules with actual bill numbers
granules_with_bills = set()
for f in ref_files:
    rdf = pd.read_csv(f)
    mask = rdf['contents__number'].notna()
    granules_with_bills.update(rdf.loc[mask, 'granuleId'].unique())

missing_granules = level1_granules - granules_with_name  # granules without committee names
recoverable = missing_granules & granules_with_bills

print(f'Granules missing policy area:         {len(missing_granules):,}')
print(f'Of those, with bill references:       {len(recoverable):,} ({len(recoverable)/len(missing_granules)*100:.1f}%)')
print()

# How many mentions would this recover?
mentions_recoverable = level1[level1['granuleId'].isin(recoverable)].shape[0]
mentions_missing = level1[level1['issue_area'].isna()].shape[0]
print(f'Mentions currently missing policy:    {mentions_missing:,}')
print(f'Mentions recoverable via bills:       {mentions_recoverable:,} ({mentions_recoverable/mentions_missing*100:.1f}%)')
print()
print('VERDICT: Bill reference fallback recovers only a small fraction.')
print('The bill data in granule_references is sparse for level1 granules.')

Granule reference files: 368


Granules missing policy area:         9,157
Of those, with bill references:       558 (6.1%)

Mentions currently missing policy:    39,502
Mentions recoverable via bills:       3,292 (8.3%)

VERDICT: Bill reference fallback recovers only a small fraction.
The bill data in granule_references is sparse for level1 granules.


### Option B: Drop Missing (Same as Thesis)

The thesis dropped ~36% of observations that had no policy area after both methods.
What happens if we do the same with the revamp?

In [10]:
# What if we just drop missing policy areas?
mentions_only = level1[level1['granuleId'].isin(level1_granules)].copy()
with_policy = mentions_only[mentions_only['issue_area'].notna()]
without_policy = mentions_only[mentions_only['issue_area'].isna()]

print('OPTION B: Drop observations without policy area')
print('=' * 60)
print(f'Total mentions:           {len(mentions_only):,}')
print(f'With policy (keep):       {len(with_policy):,} ({len(with_policy)/len(mentions_only)*100:.1f}%)')
print(f'Without policy (drop):    {len(without_policy):,} ({len(without_policy)/len(mentions_only)*100:.1f}%)')
print()

# Check if dropping creates systematic bias
print('BIAS CHECK: Are dropped mentions systematically different?')
print('-' * 60)

# Check by org category
if 'CATEGORY' in level1.columns:
    print('By org CATEGORY:')
    cat_with = with_policy.groupby('CATEGORY').size()
    cat_without = without_policy.groupby('CATEGORY').size()
    cat_compare = pd.DataFrame({'with_policy': cat_with, 'without_policy': cat_without}).fillna(0)
    cat_compare['pct_kept'] = cat_compare['with_policy'] / (cat_compare['with_policy'] + cat_compare['without_policy']) * 100
    print(cat_compare.sort_values('pct_kept').to_string())
    print()

# Check by party
if 'party' in level1.columns:
    print('By party:')
    for party in level1['party'].dropna().unique()[:5]:
        w = (with_policy['party'] == party).sum()
        wo = (without_policy['party'] == party).sum()
        t = w + wo
        if t > 0:
            print(f'  {party}: {w:,} kept / {t:,} total ({w/t*100:.1f}% retention)')
    print()

# Check by chamber
if 'chamber' in level1.columns:
    print('By chamber:')
    for ch in level1['chamber'].dropna().unique():
        w = (with_policy['chamber'] == ch).sum()
        wo = (without_policy['chamber'] == ch).sum()
        t = w + wo
        if t > 0:
            print(f'  {ch}: {w:,} kept / {t:,} total ({w/t*100:.1f}% retention)')

OPTION B: Drop observations without policy area


Total mentions:           53,892
With policy (keep):       14,390 (26.7%)
Without policy (drop):    39,502 (73.3%)

BIAS CHECK: Are dropped mentions systematically different?
------------------------------------------------------------
By org CATEGORY:
                                                                                       with_policy  without_policy   pct_kept
CATEGORY                                                                                                                     
(1005) Other foreign                                                                           0.0              10   0.000000
(1114) single issue international PIG -cons                                                    0.0               2   0.000000
(201) Corporation                                                                              0.0               3   0.000000
(701) Health institution                                                                       0.0              49  

### Option C: Fix Unmatched Committees

Some committee names in the data don't match the dictionary. Adding them helps
but won't fix the fundamental issue (most granules have no committee at all).

In [11]:
# Proposed additions for unmatched committees
PROPOSED_ADDITIONS = {
    'Ad Hoc Committee on Energy': ('Energy', 800),
    'Committee of the Whole House on the State of the Union': None,  # Procedural, skip
    'Committee on Human Resources': ('Labor', 500),
    'Committee on Internal Security': ('Law and Crime', 1200),
    'Committee on International Relations': ('International Affairs', 1900),
    'Committee on Public Works': ('Transportation', 1000),
    'Committee on Public Works and Transportation': ('Transportation', 1000),
    'Committee on Standards of Official Conduct': ('Government Operations', 2000),
    "Committee on Veterans' Affairs": ('Defense', 1600),
    'Joint Committee on Printing': ('Government Operations', 2000),
    'Joint Select Committee on Deficit Reduction': ('Macroeconomics', 100),
    'Select Committee on Assassinations': ('Law and Crime', 1200),
    'Select Committee on Energy Independence and Global Warming': ('Energy', 800),
    'Select Committee on the Events Surrounding the 2012 Terrorist Attack in Benghazi': ('International Affairs', 1900),
}

# How many additional granules would these cover?
additional_granules = set()
for f in files:
    cdf = pd.read_csv(f)
    cdf = cdf[cdf['granuleId'].isin(missing_granules) & cdf['committeeName'].notna()]
    for _, row in cdf.iterrows():
        cn = row['committeeName']
        for key, val in PROPOSED_ADDITIONS.items():
            if val is not None and key.lower() in cn.lower():
                additional_granules.add(row['granuleId'])

additional_mentions = level1[level1['granuleId'].isin(additional_granules)].shape[0]
print(f'Proposed committee additions: {sum(1 for v in PROPOSED_ADDITIONS.values() if v is not None)}')
print(f'Additional granules covered:  {len(additional_granules)}')
print(f'Additional mentions covered:  {additional_mentions}')
print()
new_total = revamp_with + additional_mentions
print(f'New coverage after additions: {new_total:,} / {revamp_total:,} ({new_total/revamp_total*100:.1f}%)')
print()
print('VERDICT: Fixing unmatched committees adds minimal coverage.')
print('The fundamental issue is that most granules have NO committee metadata at all.')

Proposed committee additions: 13
Additional granules covered:  0
Additional mentions covered:  0

New coverage after additions: 14,390 / 53,892 (26.7%)

VERDICT: Fixing unmatched committees adds minimal coverage.
The fundamental issue is that most granules have NO committee metadata at all.


## Part 4: Recommended Path Forward

Given the SEVERE coverage gap (>70% missing), here are the options ranked by feasibility:

In [12]:
print('RECOMMENDED PATH FORWARD')
print('=' * 60)
print()
print('Option 1: DROP MISSING + DOCUMENT (Recommended)')
print('-' * 60)
print('The thesis itself dropped ~36% of observations without policy areas.')
print('The revamp would drop ~73%, which is larger but the MECHANISM is the')
print('same: observations without committee metadata get no policy area.')
print()
print('Justification: The revamp covers 114th Congress only (vs. 107th-114th')
print('in the thesis). The committee metadata availability varies by Congress.')
print('The thesis\'s bill classifier was a custom external tool that is not')
print('available for the rework. Dropping observations is defensible if:')
print('  a) The drop is documented transparently')
print('  b) A bias check shows no systematic exclusion by party/chamber/etc.')
print('  c) The remaining N is sufficient for the GLMM (~14K mentions)')
print()
print(f'  Remaining mentions for GLMM: ~{revamp_with:,}')
print(f'  Legacy had (after drops):    ~{legacy_combined:,}')
print(f'  Ratio: {revamp_with/legacy_combined:.1f}x the legacy sample')
print()
print()
print('Option 2: ADD BILL POLICY AREA FALLBACK')
print('-' * 60)
print('Use Congress.gov API policyArea field as fallback for granules')
print('without committee data but with bill references.')
print(f'Expected gain: ~{mentions_recoverable:,} additional mentions')
print(f'New coverage: ~{(revamp_with + mentions_recoverable)/revamp_total*100:.1f}%')
print('Effort: Moderate (API calls + LoC-to-CAP mapping)')
print()
print()
print('Option 3: TEXT-BASED CLASSIFICATION')
print('-' * 60)
print('Train or use a pre-trained classifier on Congressional Record text')
print('to assign CAP codes. Most comprehensive but highest effort.')
print(f'Expected gain: Could recover most of the ~{revamp_total - revamp_with:,} missing mentions')
print('Effort: High (model selection, training/fine-tuning, validation)')
print()

RECOMMENDED PATH FORWARD

Option 1: DROP MISSING + DOCUMENT (Recommended)
------------------------------------------------------------
The thesis itself dropped ~36% of observations without policy areas.
The revamp would drop ~73%, which is larger but the MECHANISM is the
same: observations without committee metadata get no policy area.

Justification: The revamp covers 114th Congress only (vs. 107th-114th
in the thesis). The committee metadata availability varies by Congress.
The thesis's bill classifier was a custom external tool that is not
available for the rework. Dropping observations is defensible if:
  a) The drop is documented transparently
  b) A bias check shows no systematic exclusion by party/chamber/etc.
  c) The remaining N is sufficient for the GLMM (~14K mentions)

  Remaining mentions for GLMM: ~14,390
  Legacy had (after drops):    ~14,392
  Ratio: 1.0x the legacy sample


Option 2: ADD BILL POLICY AREA FALLBACK
-------------------------------------------------------

## Part 5: Mapping Duplication Audit

The committee-to-policy mapping is duplicated across multiple files.
This is a maintenance risk — if one copy is updated, others may drift.

In [13]:
import ast
print('COMMITTEE-POLICY MAPPING DUPLICATION')
print('=' * 60)
print()

locations = [
    ('committee_policy_linkage.py', '../../interest_group_analysis/2.data_processing/committee_policy_linkage.py'),
    ('build_analysis_dataset.py', '../../interest_group_analysis/4_integration/build_analysis_dataset.py'),
    ('derive_policy_overlap.py', '../../scripts/derive_policy_overlap.py'),
    ('multi_level_builder.py', '../../interest_group_analysis/5_analysis/multi_level_builder.py'),
]

for name, path in locations:
    try:
        with open(path, encoding='utf-8') as f:
            text = f.read()
        # Count lines that look like mapping entries
        import re
        entries = len(re.findall(r"'Committee|'Joint|'Select|'Special|'Permanent|'House|'Ad Hoc|'Commission|'United States", text))
        print(f'{name:40s}: ~{entries} committee entries')
    except FileNotFoundError:
        print(f'{name:40s}: FILE NOT FOUND')

print()
print('RECOMMENDATION:')
print('Extract mapping into data/input/committee_policy_mapping.json')
print('and have all scripts load from that single source of truth.')
print('(Flagged as future cleanup task - NOT modifying existing scripts.)')

COMMITTEE-POLICY MAPPING DUPLICATION

committee_policy_linkage.py             : ~77 committee entries
build_analysis_dataset.py               : ~43 committee entries
derive_policy_overlap.py                : ~0 committee entries
multi_level_builder.py                  : ~44 committee entries

RECOMMENDATION:
Extract mapping into data/input/committee_policy_mapping.json
and have all scripts load from that single source of truth.
(Flagged as future cleanup task - NOT modifying existing scripts.)


## Final Summary

In [14]:
print('POLICY AREA STATUS')
print('=' * 60)
print(f'Coverage in level1.csv:              {revamp_pct:.1f}%')
print(f'Coverage in analysis_dataset:        {revamp_with}/{len(df):,} ({revamp_with/len(df)*100:.1f}%)')
print(f'Legacy coverage (before drops):      {legacy_pct_before:.1f}%')
print(f'Gap-filling method used:             Committee-only (no bill fallback)')
print(f'Mentions available for GLMM:         {revamp_with:,} (after dropping missing policy areas)')
print(f'Legacy had after drops:              {legacy_combined:,}')
print(f'Policy area available for GLMM:      YES (for {revamp_pct:.0f}% of mentions)')
print()
print('ROOT CAUSE:')
print(f'  {len(level1_granules):,} unique granules in level1')
print(f'  {len(granules_with_name):,} ({len(granules_with_name)/len(level1_granules)*100:.1f}%) have committee metadata in GovInfo API')
print(f'  {len(level1_granules) - len(granules_with_name):,} ({(len(level1_granules) - len(granules_with_name))/len(level1_granules)*100:.1f}%) have no committee data at all')
print(f'  The thesis compensated with a bill classifier; the revamp does not.')
print()
print('RECOMMENDED ACTION:')
print('  1. Fix 14 unmatched committee names (trivial gain)')
print('  2. Drop observations without policy area (same as thesis)')
print('  3. Document the drop transparently in methodology')
print(f'  4. Remaining ~{revamp_with:,} mentions is {revamp_with/legacy_combined:.1f}x the legacy sample')
print('  5. Consider bill-based fallback as optional enhancement')
print()
print('MAPPING DUPLICATION:')
print('  Committee-policy dict exists in 4 files (future cleanup task).')
print('  Create data/input/committee_policy_mapping.json as single source.')

POLICY AREA STATUS
Coverage in level1.csv:              26.7%
Coverage in analysis_dataset:        14390/57,073 (25.2%)
Legacy coverage (before drops):      64.2%
Gap-filling method used:             Committee-only (no bill fallback)
Mentions available for GLMM:         14,390 (after dropping missing policy areas)
Legacy had after drops:              14,392
Policy area available for GLMM:      YES (for 27% of mentions)

ROOT CAUSE:
  10,657 unique granules in level1
  1,500 (14.1%) have committee metadata in GovInfo API
  9,157 (85.9%) have no committee data at all
  The thesis compensated with a bill classifier; the revamp does not.

RECOMMENDED ACTION:
  1. Fix 14 unmatched committee names (trivial gain)
  2. Drop observations without policy area (same as thesis)
  3. Document the drop transparently in methodology
  4. Remaining ~14,390 mentions is 1.0x the legacy sample
  5. Consider bill-based fallback as optional enhancement

MAPPING DUPLICATION:
  Committee-policy dict exists in 